In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    LSTM, Dense, Dropout, Bidirectional, GlobalAveragePooling1D,
    Conv1D, BatchNormalization, Activation, RepeatVector,
    TimeDistributed, Attention, Concatenate, AdditiveAttention,Attention,Lambda, Add)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras import layers, Model, Input

from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.metrics import r2_score, mean_absolute_error,mean_squared_error

import numpy as np, re
import pandas as pd
import matplotlib.pyplot as plt
import gc, os, time
from keras import backend as K
from vmdpy import VMD

Yt = pd.read_csv('ws.csv', header=1, parse_dates=['Timestamp'])
Yt = Yt.rename(columns={'Timestamp': 'time'})

Yt['wind_sin'] = np.sin(np.deg2rad(Yt['Ch8_Vane_10.00m_N_Avg_Deg'] % 360))
Yt['wind_cos'] = np.cos(np.deg2rad(Yt['Ch8_Vane_10.00m_N_Avg_Deg'] % 360))

# 2. 湍流强度 SD（四层）轻度 clip
sd_cols = [
    'Ch1_Anem_110.00m_E_SD_m/s',
    'Ch2_Anem_50.00m_E_SD_m/s',
    'Ch3_Anem_30.00m_E_SD_m/s',
    'Ch4_Anem_10.00m_E_SD_m/s']

def clip_sd(col):
    Q1 = Yt[col].quantile(0.25)
    Q3 = Yt[col].quantile(0.75)
    IQR = Q3 - Q1
    upper = Q3 + 3 * IQR
    Yt[col] = np.clip(Yt[col], None, upper)

for c in sd_cols:
    clip_sd(c)

# 3. 湍流强度 TI = SD / Mean
Yt['TI_110'] = Yt['Ch1_Anem_110.00m_E_SD_m/s'] / Yt['Ch1_Anem_110.00m_E_Avg_m/s']
Yt['TI_50']  = Yt['Ch2_Anem_50.00m_E_SD_m/s']  / Yt['Ch2_Anem_50.00m_E_Avg_m/s']
Yt['TI_30']  = Yt['Ch3_Anem_30.00m_E_SD_m/s']  / Yt['Ch3_Anem_30.00m_E_Avg_m/s']
Yt['TI_10']  = Yt['Ch4_Anem_10.00m_E_SD_m/s']  / Yt['Ch4_Anem_10.00m_E_Avg_m/s']

Yt.replace([np.inf, -np.inf], np.nan, inplace=True)
Yt.fillna(0, inplace=True)

# 4. 阵风偏差（gust deviation）
Yt['gust_dev_110'] = Yt['Ch1_Anem_110.00m_E_Gust_m/s'] - Yt['Ch1_Anem_110.00m_E_Avg_m/s']
Yt['gust_dev_50']  = Yt['Ch2_Anem_50.00m_E_Gust_m/s']  - Yt['Ch2_Anem_50.00m_E_Avg_m/s']
Yt['gust_dev_30']  = Yt['Ch3_Anem_30.00m_E_Gust_m/s']  - Yt['Ch3_Anem_30.00m_E_Avg_m/s']
Yt['gust_dev_10']  = Yt['Ch4_Anem_10.00m_E_Gust_m/s']  - Yt['Ch4_Anem_10.00m_E_Avg_m/s']

# 5. 层间风切变（vertical shear）
Yt['shear_110_10'] = Yt['Ch1_Anem_110.00m_E_Avg_m/s'] - Yt['Ch4_Anem_10.00m_E_Avg_m/s']
Yt['shear_110_50'] = Yt['Ch1_Anem_110.00m_E_Avg_m/s'] - Yt['Ch2_Anem_50.00m_E_Avg_m/s']
Yt['shear_50_10']  = Yt['Ch2_Anem_50.00m_E_Avg_m/s']  - Yt['Ch4_Anem_10.00m_E_Avg_m/s']
Yt['shear_30_10']  = Yt['Ch3_Anem_30.00m_E_Avg_m/s']  - Yt['Ch4_Anem_10.00m_E_Avg_m/s']

# 6. 气象变量：温度 + 气压
Yt['temp'] = Yt['Ch9_Analog_10.00m_N_Avg_C']
Yt['pressure'] = Yt['Ch10_Analog_10.00m_N_Avg_kpa']

# 7. 时间特征（分钟周期）
Yt['minute'] = Yt['time'].dt.minute
Yt['minute_sin'] = np.sin(2 * np.pi * Yt['minute'] / 60)
Yt['minute_cos'] = np.cos(2 * np.pi * Yt['minute'] / 60)

features = [

    # 原始风速
    'Ch4_Anem_10.00m_E_Avg_m/s',
    'Ch3_Anem_30.00m_E_Avg_m/s',
    'Ch2_Anem_50.00m_E_Avg_m/s',
    'Ch1_Anem_110.00m_E_Avg_m/s',

    # 风向
    'wind_sin', 'wind_cos',

    # SD turbulence
    'Ch1_Anem_110.00m_E_SD_m/s',
    'Ch2_Anem_50.00m_E_SD_m/s',
    'Ch3_Anem_30.00m_E_SD_m/s',
    'Ch4_Anem_10.00m_E_SD_m/s',

    # TI
    'TI_110', 'TI_50', 'TI_30', 'TI_10',

    # gust deviation
    'gust_dev_110', 'gust_dev_50', 'gust_dev_30', 'gust_dev_10',

    # shear
    'shear_110_10', 'shear_110_50', 'shear_50_10', 'shear_30_10',

    # 气象变量
    'temp', 'pressure',

    # 时间周期
    'minute_sin', 'minute_cos']

targets = [
    'Ch1_Anem_110.00m_E_Avg_m/s',
    'Ch2_Anem_50.00m_E_Avg_m/s',
    'Ch3_Anem_30.00m_E_Avg_m/s',
    'Ch4_Anem_10.00m_E_Avg_m/s']

K_vmd = 8
alpha_vmd = 3500

def vmd_comp(df, col, K=K_vmd, alpha=alpha_vmd):
    signal = df[col].values
    signal = signal - np.mean(signal)
    tau = 0
    DC = 0
    init = 1
    tol = 1e-7
    u, u_hat, omega = VMD(signal, alpha, tau, K, DC, init, tol)
    for k in range(K):
        imf = u[k]
        if len(imf) < len(df):
            imf = np.append(imf, imf[-1])
        df[f"{col}_IMF{k+1}"] = imf
    return df

target_h = ["Ch1_Anem_110.00m_E_Avg_m/s"]
for th in target_h:
    print("Decomposing:", th)
    Yt = vmd_comp(Yt, col=th, K=K_vmd, alpha=alpha_vmd)
    for k in range(1, K_vmd + 1):
        imf_col = f"{th}_IMF{k}"
        if imf_col not in features:
            features.append(imf_col)

Decomposing: Ch1_Anem_110.00m_E_Avg_m/s


In [ ]:
np.random.seed(42)
tf.random.set_seed(42)


def dataset_multi(X_scaled, y_real, window_size=3, pred_steps=1):
    X, y = [], []
    n = len(X_scaled)

    for i in range(n - window_size - pred_steps + 1):
        X.append(
            X_scaled[
                i:i + window_size,
                :
            ]
        )

        y.append(
            y_real[
                i + window_size:
                i + window_size + pred_steps,
                0
            ]
        )

    return np.array(X), np.array(y)


def metrics(y_true, y_pred):
    y_true = np.array(y_true).flatten()
    y_pred = np.array(y_pred).flatten()

    rmse_val = np.sqrt(
        mean_squared_error(
            y_true,
            y_pred
        )
    )

    mae_val = mean_absolute_error(
        y_true,
        y_pred
    )

    r2_val = r2_score(
        y_true,
        y_pred
    )

    mape_val = np.mean(
        np.abs(
            (y_true - y_pred)
            / np.maximum(np.abs(y_true), 1e-6)
        )
    ) * 100

    return rmse_val, mae_val, mape_val, r2_val


w_list = [3, 6, 12, 24]
f_list = [1, 2, 3, 4, 5, 6]

results = []
predictions = {}

target_list = [
    "Ch1_Anem_110.00m_E_Avg_m/s"
]


df_y = Yt[
    ['time'] + features
].copy()

df_y = df_y.sort_values(
    'time'
).reset_index(
    drop=True
)

split_idx = int(
    0.8 * len(df_y)
)

train_raw = df_y.iloc[
    :split_idx
].copy().reset_index(
    drop=True
)

test_raw = df_y.iloc[
    split_idx:
].copy().reset_index(
    drop=True
)


for th in target_list:
    print("Decomposing train:", th)

    train_raw = vmd_comp(
        train_raw,
        col=th,
        K=K_vmd,
        alpha=alpha_vmd
    )

    print("Decomposing test:", th)

    test_raw = vmd_comp(
        test_raw,
        col=th,
        K=K_vmd,
        alpha=alpha_vmd
    )


vmd_features = features.copy()

for th in target_list:
    for k in range(1, K_vmd + 1):
        imf_col = f"{th}_IMF{k}"

        if imf_col not in vmd_features:
            vmd_features.append(imf_col)


for target in target_list:

    train_df = train_raw[
        vmd_features
    ].copy()

    test_df = test_raw[
        vmd_features
    ].copy()


    y_train_real = train_raw[
        [target]
    ].values.astype(float)

    y_test_real = test_raw[
        [target]
    ].values.astype(float)


    scaler_X = MinMaxScaler()

    train_scaled = scaler_X.fit_transform(
        train_df.values.astype(float)
    )

    test_scaled = scaler_X.transform(
        test_df.values.astype(float)
    )


    scaler_y = MinMaxScaler()

    scaler_y.fit(
        y_train_real
    )


    for w in w_list:

        for pred_steps in f_list:

            forecast_min = pred_steps * 10

            print(
                f"\nTraining {target}: "
                f"w={w}, "
                f"Predict next {forecast_min} minutes"
            )


            X_train, y_train = dataset_multi(
                train_scaled,
                y_train_real,
                window_size=w,
                pred_steps=pred_steps
            )

            X_test, y_test = dataset_multi(
                test_scaled,
                y_test_real,
                window_size=w,
                pred_steps=pred_steps
            )


            y_train_s = scaler_y.transform(
                y_train.reshape(-1, 1)
            ).reshape(
                -1,
                pred_steps,
                1
            )

            y_test_s = scaler_y.transform(
                y_test.reshape(-1, 1)
            ).reshape(
                -1,
                pred_steps,
                1
            )


            time_steps = X_train.shape[1]
            n_features = X_train.shape[2]


            encoder_inputs = Input(
                shape=(
                    time_steps,
                    n_features
                )
            )


            encoder_lstm = LSTM(
                128,
                activation='tanh',
                return_sequences=True,
                return_state=True
            )

            encoder_outputs, state_h, state_c = encoder_lstm(
                encoder_inputs
            )


            decoder_inputs = RepeatVector(
                pred_steps
            )(state_h)


            decoder_lstm = LSTM(
                128,
                activation='tanh',
                return_sequences=True
            )

            decoder_outputs = decoder_lstm(
                decoder_inputs,
                initial_state=[
                    state_h,
                    state_c
                ]
            )


            attn_output = Attention()([
                decoder_outputs,
                encoder_outputs
            ])


            concat = Concatenate(
                axis=-1
            )([
                decoder_outputs,
                attn_output])


            output = TimeDistributed(Dense(1))(concat)
            model = Model(
                encoder_inputs,
                output)

            model.compile(
                optimizer='adam',
                loss='mse')


            start_time = time.time()


            history = model.fit(
                X_train,
                y_train_s,
                epochs=150,
                batch_size=128,
                validation_split=0.1,
                shuffle=False,
                callbacks=[
                    ReduceLROnPlateau(
                        monitor='val_loss',
                        factor=0.5,
                        patience=5,
                        min_lr=1e-5,
                        verbose=1
                    ),
                    EarlyStopping(
                        monitor='val_loss',
                        patience=10,
                        restore_best_weights=True,
                        verbose=1
                    )
                ],
                verbose=0
            )


            elapsed = time.time() - start_time

            print(
                f"Training time: "
                f"{elapsed:.2f} sec")


            y_train_pred_seq = model.predict(
                X_train,
                verbose=0)

            y_test_pred_seq = model.predict(
                X_test,
                verbose=0)


            y_train_pred_last = scaler_y.inverse_transform(
                y_train_pred_seq[
                    :,
                    -1,
                    :
                ].reshape(-1, 1)
            ).flatten()

            y_test_pred_last = scaler_y.inverse_transform(
                y_test_pred_seq[
                    :,
                    -1,
                    :
                ].reshape(-1, 1)).flatten()


            y_train_true_last = y_train[:,-1]
            y_test_true_last = y_test[:,-1]

            rmse_train, mae_train, mape_train, r2_train = metrics(
                y_train_true_last,
                y_train_pred_last
            )

            rmse_test, mae_test, mape_test, r2_test = metrics(y_test_true_last,y_test_pred_last)

            results.append({
                "Height": target,
                "Window": w,
                "Forecast_Steps": pred_steps,
                "Forecast_Min": forecast_min,
                "Train_RMSE": float(rmse_train),
                "Test_RMSE": float(rmse_test),
                "Train_MAE": float(mae_train),
                "Test_MAE": float(mae_test),
                "Train_MAPE": float(mape_train),
                "Test_MAPE": float(mape_test),
                "Train_R2": float(r2_train),
                "Test_R2": float(r2_test),
                "Time_sec": float(elapsed)})


            print(
                f"Train RMSE={rmse_train:.6f}, "
                f"Test RMSE={rmse_test:.6f}"
            )

            print(
                f"Train MAE ={mae_train:.6f}, "
                f"Test MAE ={mae_test:.6f}"
            )

            print(
                f"Train MAPE={mape_train:.2f}%, "
                f"Test MAPE={mape_test:.2f}%"
            )

            print(
                f"Train R2  ={r2_train:.6f}, "
                f"Test R2  ={r2_test:.6f}"
            )


            predictions[
                (
                    w,
                    pred_steps,
                    target
                )
            ] = {
                "train_true_last":
                    y_train_true_last.copy(),

                "train_pred_last":
                    y_train_pred_last.copy(),

                "test_true_last":
                    y_test_true_last.copy(),

                "test_pred_last":
                    y_test_pred_last.copy(),

                "unit": "m/s"
            }

            K.clear_session()
            del (
                model,
                X_train,
                y_train,
                X_test,
                y_test
            )

            gc.collect()

df_results = pd.DataFrame(results)
df_results = df_results.sort_values(["Forecast_Min","Window"]).reset_index(drop=True)

display(df_results)

Decomposing: Ch1_Anem_110.00m_E_Avg_m/s

Training Ch1_Anem_110.00m_E_Avg_m/s: Predict next 10 minutes
Model: "model"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_1 (InputLayer)        [(None, 24, 34)]             0         []                            
                                                                                                  
 lstm (LSTM)                 [(None, 24, 128),            83456     ['input_1[0][0]']             
                              (None, 128),                                                        
                              (None, 128)]                                                        
                                                                                                  
 repeat_vector (RepeatVecto  (None, 1, 128)               0         ['lstm[0][1]']         

2026-01-11 20:03:05.493226: W tensorflow/core/grappler/costs/op_level_cost_estimator.cc:693] Error in PredictCost() for the op: op: "Softmax" attr { key: "T" value { type: DT_FLOAT } } inputs { dtype: DT_FLOAT shape { unknown_rank: true } } device { type: "GPU" } outputs { dtype: DT_FLOAT shape { unknown_rank: true } }
2026-01-11 20:03:12.468434: W tensorflow/core/grappler/costs/op_level_cost_estimator.cc:693] Error in PredictCost() for the op: op: "Softmax" attr { key: "T" value { type: DT_FLOAT } } inputs { dtype: DT_FLOAT shape { unknown_rank: true } } device { type: "GPU" } outputs { dtype: DT_FLOAT shape { unknown_rank: true } }



Epoch 9: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.

Epoch 15: ReduceLROnPlateau reducing learning rate to 0.0002500000118743628.

Epoch 20: ReduceLROnPlateau reducing learning rate to 0.0001250000059371814.

Epoch 25: ReduceLROnPlateau reducing learning rate to 6.25000029685907e-05.

Epoch 30: ReduceLROnPlateau reducing learning rate to 3.125000148429535e-05.

Epoch 37: ReduceLROnPlateau reducing learning rate to 1.5625000742147677e-05.

Epoch 42: ReduceLROnPlateau reducing learning rate to 1e-05.
Restoring model weights from the end of the best epoch: 86.
Epoch 96: early stopping
Training time: 520.40 sec


2026-01-11 20:11:44.765893: W tensorflow/core/grappler/costs/op_level_cost_estimator.cc:693] Error in PredictCost() for the op: op: "Softmax" attr { key: "T" value { type: DT_FLOAT } } inputs { dtype: DT_FLOAT shape { unknown_rank: true } } device { type: "GPU" } outputs { dtype: DT_FLOAT shape { unknown_rank: true } }


339/339 [==============================] - 3s 7ms/step
Train RMSE=0.012959, Test RMSE=0.011641
Train R2  =0.992238, Test R2  =0.994196
Train RMSE=0.012959, Test RMSE=0.011641
Train MAE =0.009586,  Test MAE =0.008757
Train MAPE=2.34%, Test MAPE=2.23%
Train R2  =0.992238, Test R2  =0.994196


,Height,Forecast_Min,Train_RMSE,Test_RMSE,Train_MAE,Test_MAE,Train_MAPE,Test_MAPE,Train_R2,Test_R2
0,Ch1_Anem_110.00m_E_Avg_m/s,10,0.012959,0.011641,0.009586,0.008757,2.340769,2.231428,0.992238,0.994196


In [3]:
import os
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

base_dir = "6-110m_Results"
os.makedirs(base_dir, exist_ok=True)

df_results = pd.DataFrame(results)

if "Window" not in df_results.columns:
    current_w = w if 'w' in locals() else "Unknown"
    df_results["Window"] = current_w

for key, data in predictions.items():
    w_val, f_val, target_val = key[0], key[1], key[2]
    
    safe_h = re.sub(r'[^A-Za-z0-9]+', "_", target_val)
    w_dir = os.path.join(base_dir, f"w{w_val}")
    os.makedirs(w_dir, exist_ok=True)
    
    df_test = pd.DataFrame({
        "True_Test": data.get("test_true", data.get("test_true_f", [])),
        "Pred_Test": data.get("test_pred", data.get("test_pred_f", []))
    })
    
    f_min = f_val * 10 if f_val < 100 else f_val
    filename = f"{safe_h}_test_{f_min}min.csv"
    df_test.to_csv(os.path.join(w_dir, filename), index=False)

for w_val in df_results["Window"].unique():
    w_dir = os.path.join(base_dir, f"w{w_val}")
    os.makedirs(w_dir, exist_ok=True)
    df_w = df_results[df_results["Window"] == w_val]
    df_w.to_csv(os.path.join(w_dir, f"summary_w{w_val}.csv"), index=False)

print("CSV 数据保存完成！")

def plot_fixed(predictions_dict, output_dir):
    colors = ["#006699", "#b30000", "#009933", "#ff9900", "#660066", "#666600"]
    for idx, (key, data) in enumerate(predictions_dict.items()):
        w_curr, f_curr, target_curr = key[0], key[1], key[2]
        w_path = os.path.join(output_dir, f"w{w_curr}")
        
        t_val = data.get("test_true", data.get("test_true_f", np.array([])))
        p_val = data.get("test_pred", data.get("test_pred_f", np.array([])))
        
        if len(t_val) == 0: continue

        plt.figure(figsize=(6, 6))
        plt.scatter(t_val, p_val, alpha=0.3, color=colors[idx % len(colors)])
        
        diag_lim = [min(t_val.min(), p_val.min()), max(t_val.max(), p_val.max())]
        plt.plot(diag_lim, diag_lim, 'r--', label="Ideal")
        
        plt.title(f"Target: {target_curr[:10]}... (w={w_curr})")
        plt.xlabel("Actual")
        plt.ylabel("Predicted")
        plt.legend()
        
        f_min = f_curr * 10 if f_curr < 100 else f_curr
        plt.savefig(os.path.join(w_path, f"plot_{f_min}min.png"), dpi=300)
        plt.close()

plot_fixed(predictions, base_dir)
print("图表保存完成！")

CSV 数据保存完成！
图表保存完成！
